# Module 7: Advanced Policy Optimization — Algorithm Showdown

## Learning Objectives
By completing this notebook, you will:
1. Compare REINFORCE, A2C, and PPO under controlled conditions (identical architecture, hyperparameter budget)
2. Evaluate algorithms across multiple random seeds to separate luck from genuine performance
3. Interpret sample efficiency curves: reward per environment step rather than per episode
4. Construct a statistical comparison using mean, standard deviation, and bootstrap confidence intervals

## Prerequisites
- `01_reinforce_from_scratch.ipynb` (Module 6) — REINFORCE algorithm
- `02_actor_critic.ipynb` (Module 6) — Actor-Critic
- `01_ppo_from_scratch.ipynb` (Module 7) — PPO-Clip with GAE

## Estimated Time: 15 minutes

---

## Fair Comparison Rules

Comparing RL algorithms is notoriously difficult because results depend heavily on:
- Network architecture and capacity
- Hyperparameter tuning effort
- Number of environment steps (not just episodes)
- Random seed variance

Our comparison rules:
1. **Same network architecture**: 128-unit hidden layer, same activations
2. **Same total environment steps**: 80,000 steps per algorithm per seed
3. **Minimal hyperparameter tuning**: each algorithm uses its well-known defaults
4. **5 random seeds**: enough to estimate variance
5. **Sample efficiency metric**: reward vs. environment steps (not episodes)

In [ ]:
learning_objectives([
    "`01_reinforce_from_scratch.ipynb` (Module 6) — REINFORCE algorithm",
    "`02_actor_critic.ipynb` (Module 6) — Actor-Critic",
    "`01_ppo_from_scratch.ipynb` (Module 7) — PPO-Clip with GAE",
    "## Fair Comparison Rules",
    "Network architecture and capacity",
    "Hyperparameter tuning effort",
    "Number of environment steps (not just episodes)",
    "Random seed variance"
])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import all dependencies
# Key Concept: self-contained implementations so this notebook runs standalone

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
import warnings
from typing import List

warnings.filterwarnings('ignore')

SEEDS = [42, 43, 44, 45, 46]
TOTAL_STEPS = 80_000
HIDDEN_DIM = 128

env_probe = gym.make('CartPole-v1')
OBS_DIM = env_probe.observation_space.shape[0]
ACT_DIM = env_probe.action_space.n
env_probe.close()

print(f"Running comparison: {len(SEEDS)} seeds x 3 algorithms x {TOTAL_STEPS:,} steps")

## 1. Shared Network Architecture

All three algorithms use the same 128-unit hidden layer with Tanh activation. This eliminates capacity as a confound.

In [ ]:
# Purpose: Define shared architecture used by all three algorithms
# Key Concept: Identical capacity ensures performance differences are algorithmic

class PolicyNet(nn.Module):
    """Softmax policy network (used by REINFORCE and A2C actor)."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(OBS_DIM, HIDDEN_DIM),
            nn.Tanh(),
            nn.Linear(HIDDEN_DIM, ACT_DIM)
        )

    def forward(self, x: torch.Tensor) -> Categorical:
        return Categorical(logits=self.net(x))


class ValueNet(nn.Module):
    """Value function network (used by REINFORCE baseline and A2C critic)."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(OBS_DIM, HIDDEN_DIM),
            nn.Tanh(),
            nn.Linear(HIDDEN_DIM, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


class PPONet(nn.Module):
    """Shared-trunk actor-critic for PPO."""

    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(OBS_DIM, HIDDEN_DIM),
            nn.Tanh()
        )
        self.actor_head = nn.Linear(HIDDEN_DIM, ACT_DIM)
        self.critic_head = nn.Linear(HIDDEN_DIM, 1)

    def forward(self, x: torch.Tensor) -> tuple:
        f = self.trunk(x)
        return Categorical(logits=self.actor_head(f)), self.critic_head(f).squeeze(-1)

    def act(self, x: torch.Tensor) -> tuple:
        with torch.no_grad():
            dist, val = self.forward(x)
        a = dist.sample()
        return a, dist.log_prob(a), val


print("Network architectures defined.")

## 2. REINFORCE Trainer

REINFORCE with value baseline. Returns a list of `(step, reward)` pairs for sample efficiency curves.

In [ ]:
# Purpose: REINFORCE training loop tracking steps for sample efficiency
# Key Concept: We track (cumulative_steps, episode_reward) not (episode_number, reward)

def train_reinforce(total_steps: int = TOTAL_STEPS, lr: float = 1e-3,
                    gamma: float = 0.99, seed: int = 42) -> List[tuple]:
    """
    Train REINFORCE with baseline.

    Returns
    -------
    List of (step, episode_reward) tuples.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make('CartPole-v1')
    policy = PolicyNet()
    value = ValueNet()
    pol_opt = optim.Adam(policy.parameters(), lr=lr)
    val_opt = optim.Adam(value.parameters(), lr=lr)

    step_reward_pairs = []
    total_env_steps = 0

    while total_env_steps < total_steps:
        obs, _ = env.reset()
        log_probs, rewards_ep, obs_list = [], [], []
        done = False

        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            obs_list.append(obs_t)
            dist = policy(obs_t.unsqueeze(0))
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            obs, r, term, trunc, _ = env.step(action.item())
            rewards_ep.append(r)
            done = term or trunc
            total_env_steps += 1

        ep_reward = sum(rewards_ep)
        step_reward_pairs.append((total_env_steps, ep_reward))

        # Compute returns
        G, returns = 0.0, []
        for r in reversed(rewards_ep):
            G = r + gamma * G
            returns.insert(0, G)
        returns_t = torch.tensor(returns, dtype=torch.float32)
        returns_norm = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

        obs_stack = torch.stack(obs_list)
        values = value(obs_stack)
        advantages = returns_norm - values.detach()
        log_probs_t = torch.stack(log_probs)

        pol_loss = -(log_probs_t * advantages).sum()
        val_loss = nn.functional.mse_loss(values, returns_norm)

        pol_opt.zero_grad(); pol_loss.backward(); pol_opt.step()
        val_opt.zero_grad(); val_loss.backward(); val_opt.step()

    env.close()
    return step_reward_pairs


print("REINFORCE trainer defined.")

## 3. A2C Trainer

A2C (Advantage Actor-Critic) is a synchronous version of A3C. It uses n-step returns rather than one-step TD, giving a better bias-variance tradeoff than one-step actor-critic.

In [ ]:
# Purpose: A2C training loop with n-step returns
# Key Concept: A2C batches n steps before updating, unlike one-step actor-critic

def train_a2c(total_steps: int = TOTAL_STEPS, lr: float = 7e-4,
              gamma: float = 0.99, n_steps: int = 5,
              vf_coef: float = 0.5, ent_coef: float = 0.01,
              seed: int = 42) -> List[tuple]:
    """
    Train A2C with n-step returns.

    Returns
    -------
    List of (step, episode_reward) tuples.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make('CartPole-v1')
    policy = PolicyNet()
    value = ValueNet()
    # A2C uses a single optimizer over both networks
    params = list(policy.parameters()) + list(value.parameters())
    optimizer = optim.RMSprop(params, lr=lr, alpha=0.99, eps=1e-5)

    step_reward_pairs = []
    total_env_steps = 0
    obs, _ = env.reset()
    ep_reward = 0.0

    while total_env_steps < total_steps:
        # Collect n_steps transitions
        obs_buf, act_buf, rew_buf, val_buf, done_buf, logp_buf = [], [], [], [], [], []

        for _ in range(n_steps):
            obs_t = torch.tensor(obs, dtype=torch.float32)
            dist = policy(obs_t.unsqueeze(0))
            val = value(obs_t.unsqueeze(0))
            action = dist.sample()

            next_obs, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            ep_reward += reward
            total_env_steps += 1

            obs_buf.append(obs_t)
            act_buf.append(action)
            rew_buf.append(reward)
            val_buf.append(val.squeeze())
            done_buf.append(done)
            logp_buf.append(dist.log_prob(action))

            obs = next_obs
            if done:
                step_reward_pairs.append((total_env_steps, ep_reward))
                ep_reward = 0.0
                obs, _ = env.reset()
            if total_env_steps >= total_steps:
                break

        # Bootstrap from last state
        with torch.no_grad():
            if done_buf[-1]:
                R = 0.0
            else:
                next_obs_t = torch.tensor(obs, dtype=torch.float32)
                R = value(next_obs_t.unsqueeze(0)).item()

        # Compute n-step returns
        returns_buf = []
        for r, d in zip(reversed(rew_buf), reversed(done_buf)):
            R = r + gamma * R * (1.0 - float(d))
            returns_buf.insert(0, R)

        returns_t = torch.tensor(returns_buf, dtype=torch.float32)
        values_t = torch.stack(val_buf)
        log_probs_t = torch.stack(logp_buf)

        advantages = returns_t - values_t.detach()
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # Recompute entropy for bonus
        obs_stack = torch.stack(obs_buf)
        dist_new = policy(obs_stack)
        entropy = dist_new.entropy().mean()

        pol_loss = -(log_probs_t * advantages).mean()
        val_loss = nn.functional.mse_loss(values_t, returns_t)
        total_loss = pol_loss + vf_coef * val_loss - ent_coef * entropy

        optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(params, max_norm=0.5)
        optimizer.step()

    env.close()
    return step_reward_pairs


print("A2C trainer defined.")

## 4. PPO Trainer

The PPO implementation from notebook 01, adapted to track step-based metrics.

In [ ]:
# Purpose: PPO training loop tracking (step, reward) pairs
# Key Concept: Reusing PPO from notebook 01, now reporting per-step rather than per-episode

def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0.0
    values_ext = values + [next_value]
    for t in reversed(range(len(rewards))):
        mask = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values_ext[t + 1] * mask - values_ext[t]
        gae = delta + gamma * lam * mask * gae
        advantages.insert(0, gae)
    advantages = torch.tensor(advantages, dtype=torch.float32)
    returns = advantages + torch.tensor(values, dtype=torch.float32)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return advantages, returns


def train_ppo(total_steps: int = TOTAL_STEPS, rollout_steps: int = 512,
              lr: float = 3e-4, gamma: float = 0.99, lam: float = 0.95,
              epsilon: float = 0.2, n_epochs: int = 4,
              seed: int = 42) -> List[tuple]:
    """
    Train PPO-Clip on CartPole-v1.

    Returns
    -------
    List of (step, episode_reward) tuples.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make('CartPole-v1')
    net = PPONet()
    optimizer = optim.Adam(net.parameters(), lr=lr, eps=1e-5)

    step_reward_pairs = []
    total_env_steps = 0
    ep_reward = 0.0

    obs_buf, act_buf, logp_buf, rew_buf, val_buf, done_buf = [], [], [], [], [], []
    obs, _ = env.reset()

    while total_env_steps < total_steps:
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        action, log_prob, value = net.act(obs_t)
        next_obs, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        ep_reward += reward
        total_env_steps += 1

        obs_buf.append(obs)
        act_buf.append(action.item())
        logp_buf.append(log_prob.item())
        rew_buf.append(reward)
        val_buf.append(value.item())
        done_buf.append(done)

        obs = next_obs
        if done:
            step_reward_pairs.append((total_env_steps, ep_reward))
            ep_reward = 0.0
            obs, _ = env.reset()

        if len(obs_buf) == rollout_steps:
            with torch.no_grad():
                next_obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                _, next_val = net(next_obs_t)
                next_val = next_val.item() * (1.0 - float(done))

            advantages, returns = compute_gae(rew_buf, val_buf, done_buf, next_val, gamma, lam)
            obs_t2 = torch.tensor(np.array(obs_buf), dtype=torch.float32)
            act_t = torch.tensor(act_buf, dtype=torch.long)
            old_lp = torch.tensor(logp_buf, dtype=torch.float32)

            for _ in range(n_epochs):
                dist, values = net(obs_t2)
                new_lp = dist.log_prob(act_t)
                ratio = torch.exp(new_lp - old_lp)
                surr1 = ratio * advantages
                surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
                pol_loss = -torch.min(surr1, surr2).mean()
                val_loss = nn.functional.mse_loss(values, returns)
                ent_loss = -dist.entropy().mean()
                loss = pol_loss + 0.5 * val_loss + 0.01 * ent_loss
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                optimizer.step()

            obs_buf, act_buf, logp_buf, rew_buf, val_buf, done_buf = [], [], [], [], [], []

    env.close()
    return step_reward_pairs


print("PPO trainer defined.")

## 5. Running the Multi-Seed Comparison

We run each algorithm with 5 seeds. This takes 1-2 minutes — the progress prints help you track status.

In [ ]:
# Purpose: Run multi-seed comparison across all three algorithms
# Key Concept: Multiple seeds reveal variance — a lucky seed can make any algorithm look good

all_results = {'REINFORCE': [], 'A2C': [], 'PPO': []}

for seed in SEEDS:
    print(f"Seed {seed}: ", end='', flush=True)

    print("REINFORCE...", end=' ', flush=True)
    all_results['REINFORCE'].append(train_reinforce(TOTAL_STEPS, seed=seed))

    print("A2C...", end=' ', flush=True)
    all_results['A2C'].append(train_a2c(TOTAL_STEPS, seed=seed))

    print("PPO...")
    all_results['PPO'].append(train_ppo(TOTAL_STEPS, seed=seed))

print("\nAll runs complete.")
for algo, runs in all_results.items():
    all_final = [r[-1][1] if r else 0 for r in runs]
    print(f"  {algo}: final ep reward = {np.mean(all_final):.1f} +/- {np.std(all_final):.1f}")

## 6. Computing Sample Efficiency Curves

To plot reward vs. environment steps on a common x-axis, we interpolate each run's rewards onto a fixed grid of step values.

In [ ]:
# Purpose: Interpolate sparse episode data onto a uniform step grid
# Key Concept: Interpolation allows computing mean/std across runs with different episode counts

def interpolate_rewards(step_reward_pairs: List[tuple], grid: np.ndarray) -> np.ndarray:
    """
    Interpolate episode rewards onto a common step grid.

    Parameters
    ----------
    step_reward_pairs : list of (step, reward)
        Raw training data.
    grid : np.ndarray
        Common x-axis steps.

    Returns
    -------
    np.ndarray
        Reward interpolated at each grid point.
    """
    if not step_reward_pairs:
        return np.zeros_like(grid, dtype=float)

    steps, rewards = zip(*step_reward_pairs)
    steps = np.array(steps, dtype=float)
    rewards = np.array(rewards, dtype=float)

    # Prepend origin so interpolation starts at 0
    steps = np.concatenate([[0], steps])
    rewards = np.concatenate([[0], rewards])

    return np.interp(grid, steps, rewards)


# Build common step grid
step_grid = np.linspace(0, TOTAL_STEPS, 200)

# Compute per-algorithm mean and std across seeds
interp_results = {}
for algo, runs in all_results.items():
    curves = np.array([interpolate_rewards(run, step_grid) for run in runs])
    interp_results[algo] = {
        'mean': curves.mean(axis=0),
        'std': curves.std(axis=0),
        'all': curves
    }

print("Interpolated sample efficiency curves computed.")
print(f"Grid: {len(step_grid)} points from 0 to {TOTAL_STEPS:,} steps")

## 7. Visualization: Full Comparison

In [ ]:
# Purpose: Generate the full algorithm comparison visualization
# Key Concept: Shaded regions show variance across seeds — overlapping bands mean no significant difference

COLORS = {'REINFORCE': 'steelblue', 'A2C': 'darkorange', 'PPO': 'crimson'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Sample efficiency curves (mean ± std) ---
ax = axes[0]
for algo, data in interp_results.items():
    mean = data['mean']
    std = data['std']
    color = COLORS[algo]
    ax.plot(step_grid, mean, label=algo, color=color, linewidth=2)
    ax.fill_between(step_grid, mean - std, mean + std, color=color, alpha=0.2)

ax.axhline(y=475, color='green', linestyle='--', alpha=0.7, label='Near-optimal')
ax.set_xlabel('Environment Steps', fontsize=12)
ax.set_ylabel('Episode Reward', fontsize=12)
ax.set_title('Sample Efficiency (mean ± std, 5 seeds)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Plot 2: Final performance distribution (violin or box plot) ---
ax = axes[1]
final_rewards = {}
for algo, runs in all_results.items():
    # Average of last 5 episodes per seed
    final_rewards[algo] = [
        np.mean([r[1] for r in run[-5:]]) if len(run) >= 5 else 0
        for run in runs
    ]

algos = list(final_rewards.keys())
data_for_box = [final_rewards[a] for a in algos]
bp = ax.boxplot(data_for_box, labels=algos, patch_artist=True)
for patch, algo in zip(bp['boxes'], algos):
    patch.set_facecolor(COLORS[algo])
    patch.set_alpha(0.7)

# Overlay individual seed points
for i, (algo, vals) in enumerate(final_rewards.items()):
    x_jitter = np.random.normal(i + 1, 0.05, len(vals))
    ax.scatter(x_jitter, vals, color=COLORS[algo], zorder=5, s=40)

ax.axhline(y=475, color='green', linestyle='--', alpha=0.7)
ax.set_ylabel('Final Episode Reward (last 5 eps avg)', fontsize=11)
ax.set_title('Final Performance Distribution', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# --- Plot 3: Steps to 400 reward threshold ---
ax = axes[2]
THRESHOLD = 400

steps_to_threshold = {algo: [] for algo in COLORS}
for algo, runs in all_results.items():
    for run in runs:
        # Use 10-episode rolling average to determine when threshold is crossed
        reached = None
        rewards_list = [r[1] for r in run]
        steps_list = [r[0] for r in run]
        for i in range(10, len(rewards_list)):
            if np.mean(rewards_list[i-10:i]) >= THRESHOLD:
                reached = steps_list[i]
                break
        steps_to_threshold[algo].append(reached if reached else TOTAL_STEPS)

for i, (algo, vals) in enumerate(steps_to_threshold.items()):
    x_jitter = np.random.normal(i + 1, 0.06, len(vals))
    ax.scatter(x_jitter, vals, color=COLORS[algo], s=60, zorder=5, label=algo)
    ax.hlines(np.mean(vals), i + 0.7, i + 1.3, color=COLORS[algo], linewidth=3)

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(list(steps_to_threshold.keys()))
ax.set_ylabel(f'Steps to Reach {THRESHOLD} Reward (10-ep avg)', fontsize=11)
ax.set_title('Sample Efficiency: Steps to Threshold', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('algorithm_showdown.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to algorithm_showdown.png")

## 8. Statistical Summary

In [ ]:
# Purpose: Print a clean statistical summary table
# Key Concept: Report mean, std, min, and max across seeds — never report a single seed result

print("=" * 60)
print(f"{'Algorithm':<12} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 60)

for algo in ['REINFORCE', 'A2C', 'PPO']:
    vals = final_rewards[algo]
    print(f"{algo:<12} {np.mean(vals):>8.1f} {np.std(vals):>8.1f} "
          f"{np.min(vals):>8.1f} {np.max(vals):>8.1f}")

print("=" * 60)
print("(Final performance = mean of last 5 episodes per seed)")

print("\nSteps to threshold ({} reward, 10-ep avg):".format(THRESHOLD))
print("-" * 60)
for algo in ['REINFORCE', 'A2C', 'PPO']:
    vals = steps_to_threshold[algo]
    reached = [v for v in vals if v < TOTAL_STEPS]
    n_reached = len(reached)
    avg_steps = np.mean(reached) if reached else TOTAL_STEPS
    print(f"{algo:<12} reached: {n_reached}/{len(SEEDS)} seeds | "
          f"avg steps: {avg_steps:,.0f}")

## Exercise 7.3: Bootstrap Confidence Intervals

**Task:** Compute 95% bootstrap confidence intervals for the final mean reward of each algorithm. Bootstrap: resample the 5 seeds with replacement 1000 times, compute the mean each time, then take the 2.5th and 97.5th percentiles.

**Expected output:** Three tuples `(lower, upper)` — one per algorithm.

<details>
<summary>Hint 1</summary>
Use `np.random.choice(vals, size=len(vals), replace=True)` in a loop.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Store 1000 bootstrap means in a list, then use `np.percentile(means, [2.5, 97.5])`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
np.random.seed(0)

def bootstrap_ci(values: list, n_boot: int = 1000, ci: float = 95.0) -> tuple:
    """
    Compute bootstrap confidence interval for the mean.

    Parameters
    ----------
    values : list of float
        Observed values (one per seed).
    n_boot : int
        Number of bootstrap resamples.
    ci : float
        Confidence level (e.g., 95.0 for 95% CI).

    Returns
    -------
    tuple of (lower, upper) bounds.
    """
    your_answer = None  # Replace with your implementation
    return your_answer


ci_results = {}  # Map algo -> (lower, upper)
# Fill ci_results and print a table

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_7_3():
    # Test bootstrap_ci on known data
    np.random.seed(0)
    data = [100.0, 100.0, 100.0, 100.0, 100.0]
    lower, upper = bootstrap_ci(data)
    assert lower is not None and upper is not None, \
        "bootstrap_ci must return a (lower, upper) tuple, not None"
    assert abs(lower - 100.0) < 0.1, f"For constant data, lower CI should be 100, got {lower}"
    assert abs(upper - 100.0) < 0.1, f"For constant data, upper CI should be 100, got {upper}"

    # Test that CI is wider for higher-variance data
    np.random.seed(0)
    data_narrow = [50.0, 51.0, 50.0, 51.0, 50.0]
    data_wide = [10.0, 90.0, 10.0, 90.0, 50.0]
    lo_n, hi_n = bootstrap_ci(data_narrow)
    lo_w, hi_w = bootstrap_ci(data_wide)
    assert (hi_w - lo_w) > (hi_n - lo_n), \
        "High-variance data should produce a wider CI"

    assert len(ci_results) == 3, \
        f"ci_results should have 3 algorithms, got {len(ci_results)}"
    for algo in ['REINFORCE', 'A2C', 'PPO']:
        assert algo in ci_results, f"Missing CI for {algo}"

    print("Exercise 7.3 passed! Bootstrap confidence intervals correct.")

test_exercise_7_3()

## Exercise 7.4: Analyzing the Impact of n_steps in A2C

**Task:** A2C's `n_steps` controls how many environment steps are collected before each update. Try `n_steps` in `{1, 5, 20, 100}`. Train each for 60,000 steps with 2 seeds each. Report the final mean reward.

**Expected observation:** Very small n_steps (1) should behave like one-step actor-critic; very large n_steps approaches REINFORCE. The sweet spot is typically 5-20.

<details>
<summary>Hint 1</summary>
Call `train_a2c(total_steps=60_000, n_steps=n, seed=s)` in a nested loop.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
n_steps_values = [1, 5, 20, 100]
n_steps_results = {}  # Map n_steps -> mean_final_reward (across 2 seeds)

# Fill n_steps_results
# Print a table and plot results

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_7_4():
    assert len(n_steps_results) == 4, \
        f"Expected 4 n_steps experiments, got {len(n_steps_results)}"
    for n in n_steps_values:
        assert n in n_steps_results, f"Missing result for n_steps={n}"
        val = n_steps_results[n]
        assert isinstance(val, (int, float)), \
            f"n_steps_results[{n}] should be scalar, got {type(val)}"
        assert val >= 0, f"Reward must be non-negative, got {val}"
    print("Exercise 7.4 passed! A2C n_steps sensitivity analysis complete.")

test_exercise_7_4()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **PPO is consistently the strongest** on CartPole at 80k steps — not because of any single trick, but because clipping + GAE + multiple epochs gives better sample efficiency than simpler algorithms.

2. **A2C beats REINFORCE in sample efficiency** because n-step updates are more informative than waiting for episode termination.

3. **Seed variance is large** — a single REINFORCE run can easily outperform a single PPO run by luck. Always report results across multiple seeds.

4. **Report steps, not episodes** — algorithms that take shorter episodes make more updates per episode, inflating episode-based metrics. Environment steps are the fair currency.

5. **Bootstrap CIs reveal overlap** between algorithms. Overlapping 95% CIs on 5 seeds does not mean the algorithms are identical — it just means we don't have enough data to be statistically confident.

## What's Next

Module 8 takes a fundamentally different approach: instead of improving how we optimize the policy, we improve how we use experience by learning a **model of the environment**. Dyna-Q uses a learned model to generate synthetic experience, dramatically improving sample efficiency.

## Additional Resources

- Henderson et al. (2018): "Deep Reinforcement Learning That Matters" — the definitive paper on RL reproducibility
- Mnih et al. (2016): "Asynchronous Methods for Deep Reinforcement Learning" — A3C
- Stable-Baselines3: production-quality implementations for benchmarking

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])